In [ ]:
import json
from sklearn.metrics import roc_auc_score
import numpy as np
import os

root_dir = "res/20251109-161812/"

with open(os.path.join(root_dir, "disturb_caption_1/embedding/attack_results.json"), "r") as f:
    disturb_1 = json.load(f)

with open(os.path.join(root_dir, "disturb_caption_2/embedding/attack_results.json"), "r") as f:
    disturb_2 = json.load(f)

with open(os.path.join(root_dir, "disturb_caption_3/embedding/attack_results.json"), "r") as f:
    disturb_3 = json.load(f)

auc_list = []
best_k = ()

y_true = []
disturb1_scores = []
disturb2_scores = []
disturb3_scores = []
save_data = []

for disturb1_entry in disturb_1:
    disturbed1_similarity = None
    disturbed2_similarity = None
    disturbed3_similarity = None
    for disturb2_entry in disturb_2:
        if disturb1_entry['path'] == disturb2_entry['path']:
            disturbed1_similarity = np.sort(disturb1_entry['disturbed_similarity'])
            disturbed2_similarity = np.sort(disturb2_entry['disturbed_similarity'])
            break
    for disturb3_entry in disturb_3:
        if disturb1_entry['path'] == disturb3_entry['path']:
            disturbed3_similarity = np.sort(disturb3_entry['disturbed_similarity'])
            break
    if disturbed1_similarity is not None and disturbed2_similarity is not None and disturbed3_similarity is not None:
        disturb1_scores.append(np.mean(disturbed1_similarity[-1:]))
        disturb2_scores.append(np.mean(disturbed2_similarity[-1:]))
        disturb3_scores.append(np.mean(disturbed3_similarity[-1:]))
        y_true.append(disturb1_entry['label'])
        save_data.append({
            "path": disturb1_entry['path'],
            "label": disturb1_entry['label'],
            "disturb1_score": np.mean(disturbed1_similarity[-1:]),
            "disturb2_score": np.mean(disturbed2_similarity[-1:]),
            "disturb3_score": np.mean(disturbed3_similarity[-1:]),
        })

In [ ]:
disturb1_scores = (disturb1_scores - np.min(disturb1_scores)) / (np.max(disturb1_scores) - np.min(disturb1_scores))
disturb2_scores = (disturb2_scores - np.min(disturb2_scores)) / (np.max(disturb2_scores) - np.min(disturb2_scores))
disturb3_scores = (disturb3_scores - np.min(disturb3_scores)) / (np.max(disturb3_scores) - np.min(disturb3_scores))

In [ ]:
np.random.seed(42)

selected_indices = np.random.choice(len(y_true), size=len(y_true)//4, replace=False)

train_y_true = np.array([y_true[i] for i in range(len(y_true)) if i in selected_indices])
train_disturb1_scores = np.array([disturb1_scores[i] for i in range(len(disturb1_scores)) if i in selected_indices])
train_disturb2_scores = np.array([disturb2_scores[i] for i in range(len(disturb2_scores)) if i in selected_indices])
train_disturb3_scores = np.array([disturb3_scores[i] for i in range(len(disturb3_scores)) if i in selected_indices])

eval_y_true = np.array([y_true[i] for i in range(len(y_true)) if i not in selected_indices])
eval_disturb1_scores = np.array([disturb1_scores[i] for i in range(len(disturb1_scores)) if i not in selected_indices])
eval_disturb2_scores = np.array([disturb2_scores[i] for i in range(len(disturb2_scores)) if i not in selected_indices])
eval_disturb3_scores = np.array([disturb3_scores[i] for i in range(len(disturb3_scores)) if i not in selected_indices])

In [ ]:
weight1 = np.linspace(0, 1, 20)
weight2 = np.linspace(0, 1, 20)
weight3 = np.linspace(0, 1, 20)

auc_list = []
best_k = ()

for w1 in weight1:
    for w2 in weight2:
        y_scores = []
        for d1, d2, d3 in zip(train_disturb1_scores, train_disturb2_scores, train_disturb3_scores):
            y_scores.append(w1 * d1 + w2 * d2 + d3)
        auc = roc_auc_score(train_y_true, y_scores)
        auc_list.append(auc)
        if auc == max(auc_list):
            best_k = (w1, w2, )

print("Best weight:", best_k)
print("Best train AUC:", max(auc_list))

In [ ]:
from src.result_process import compute_subsampled_metrics,compute_auc

results = compute_subsampled_metrics(
    label=eval_y_true,
    scores=np.array(best_k[0]) * np.array(eval_disturb1_scores)
    + np.array(best_k[1]) * np.array(eval_disturb2_scores)
    + np.array(eval_disturb3_scores),
    n_samples_per_class=300,
    n_repeats=5,
)
print("\n" + "=" * 60)
print(f"standard AUC (full data): {results['stand_auc_full_data']*100:.2f}")
print("-" * 60)
print(f"mean AUC:          {results['auc_mean']*100:.2f} (std: {results['auc_std']*100:.2f})")
print(f"mean TPR@1% FPR:   {results['tpr@1%_mean']*100:.2f} (std: {results['tpr@1%_std']*100:.2f})")
print(f"mean TPR@5% FPR:   {results['tpr@5%_mean']*100:.2f} (std: {results['tpr@5%_std']*100:.2f})")
print("=" * 60)